# Evaluación con test

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import joblib

from preprocessing import AmesFeatureEngineer

saved = joblib.load('models/preprocessing_pipeline.joblib') 

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando device: {device}")

X_train = np.load('data/X_train_processed.npy')
X_val = np.load('data/X_val_processed.npy')
y_train = np.load('data/y_train.npy')   
y_val = np.load('data/y_val.npy')

Usando device: cpu


In [2]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout=0.0, batch_norm=False):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            if batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_model(hidden_layers, hp, X_train, y_train, X_val, y_val, verbose=True):
    model = MLP(X_train.shape[1], hidden_layers,
                dropout=hp['dropout'], batch_norm=hp['batch_norm']).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=hp['lr'], weight_decay=hp['weight_decay'])
    loss_fn = nn.MSELoss()

    Xtr_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    Xva_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    yva_t = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=hp['batch_size'], shuffle=True)

    history = {'train_rmse': [], 'val_rmse': []}
    best_val_rmse = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(hp['max_epochs']):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            train_rmse = torch.sqrt(loss_fn(model(Xtr_t), ytr_t)).item()
            val_rmse = torch.sqrt(loss_fn(model(Xva_t), yva_t)).item()
        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if verbose and epoch % 20 == 0:
            print(f"  epoch {epoch:3d}  train={train_rmse:.4f}  val={val_rmse:.4f}")

        if epochs_no_improve >= hp['patience']:
            if verbose:
                print(f"  Early stopping en epoch {epoch} (best val_rmse={best_val_rmse:.4f})")
            break

    model.load_state_dict(best_state)
    return model, history, best_val_rmse


BEST_HP = {
    'lr': 5e-4, 'dropout': 0.1, 'weight_decay': 1e-4,
    'batch_size': 32, 'max_epochs': 200, 'patience': 15, 'batch_norm': True,
}
FIXED_ARCH = [256, 128, 64, 32]

## Train del modelo de evaluación

In [3]:
eval_model, eval_history, eval_val_rmse_log = train_model(
    FIXED_ARCH, BEST_HP, X_train, y_train, X_val, y_val
)
print(f"\nVal RMSE (log): {eval_val_rmse_log:.4f}")

  epoch   0  train=11.6462  val=11.6406
  epoch  20  train=4.6216  val=4.7007
  epoch  40  train=0.6568  val=0.6958
  epoch  60  train=0.3784  val=0.4139
  epoch  80  train=0.3915  val=0.4128
  Early stopping en epoch 88 (best val_rmse=0.3320)

Val RMSE (log): 0.3320


## RMSE

In [4]:
eval_model.eval()
with torch.no_grad():
    val_pred_log = eval_model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy()

val_pred_usd = np.expm1(val_pred_log)
val_true_usd = np.expm1(y_val)

rmse_usd = np.sqrt(np.mean((val_pred_usd - val_true_usd) ** 2))
mae_usd = np.mean(np.abs(val_pred_usd - val_true_usd))
mape = np.mean(np.abs((val_pred_usd - val_true_usd) / val_true_usd)) * 100

print(f"RMSE (USD):  ${rmse_usd:,.2f}")
print(f"MAE  (USD):  ${mae_usd:,.2f}")
print(f"MAPE:        {mape:.2f}%")
print(f"SalePrice medio en validación: ${val_true_usd.mean():,.2f}")

RMSE (USD):  $65,173.87
MAE  (USD):  $47,181.79
MAPE:        23.57%
SalePrice medio en validación: $183,066.07
